Copyright 2026 Google LLC

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a target="_blank" href="https://colab.research.google.com/github/google-gemma/gbench/blob/main/examples/notebooks/04_academic_evaluations_and_reasoning.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# gbench academic evaluations, reasoning mode, and multimodal budgets

**Author:** [Luciano Martins](https://github.com/lucianommartins)

This notebook explores how to measure the reasoning accuracy of foundation models using `gbench --evals`. You will run native Python evaluation harnesses for science, mathematics, and multimodal understanding, configuring Chain-of-Thought reasoning (`--eval-thinking`) and vision soft-token budgets.

You can point `gbench` at **any OpenAI-compatible `/v1` endpoint** via `--remote-endpoint`. This notebook drives a **local Ollama server** serving a quantized Gemma 4 GGUF, and also shows the **vLLM remote-endpoint** path (`--remote-endpoint http://127.0.0.1:8000/v1 --tokenizer google/gemma-4-26B-A4B-it`, served model id `google/gemma-4-26B-A4B-it`) alongside it.

## Learning objectives

1. Configure Ollama to serve a quantized Google Gemma 4 model (`unsloth/gemma-4-E4B-it-qat-GGUF`) with a context window of 40960 tokens (the server window bounds prompt + reply; `mmlu_pro` alone asks for up to 32768 reply tokens).
2. Execute native Python academic evaluation suites (`gpqa_diamond`, `mmlu_pro`, `mmmu_pro`) against an OpenAI-compatible endpoint.
3. Enable Chain-of-Thought reasoning mode (`--eval-thinking`) on thinking-aware suites and analyze its impact on reasoning accuracy.
4. Provide the required `--max-output-tokens` budget (65536 with thinking on), and serve a context window that holds prompt + budget (Ollama `num_ctx`, vLLM `--max-model-len`), so long reasoning chains are not truncated.
5. Load the per-suite `eval_<suite>_<model>_<format>.json` result files to inspect scores.
6. Run a multimodal vision suite (`mmmu_pro`) and understand soft-token budgeting (server-side when using `--remote-endpoint`).
7. Perform a clean session shutdown to terminate background servers and reclaim hardware memory.

## Useful resources

* [gbench GitHub repository](https://www.github.com/google-gemma/gbench)
* [GPQA science evaluation benchmark](https://github.com/idavidrein/gpqa)
* [MMLU-Pro reasoning benchmark](https://huggingface.co/datasets/TIGER-Lab/MMLU-Pro)
* [MMMU-Pro multimodal benchmark](https://mmmu-benchmark.github.io/)

## 1. Environment setup and installation

We use the `gbench` checkout this notebook is in (or clone the repository from GitHub), change directory into the project root, and install the package in editable mode with the `evals` extra plus `transformers` (`pip install -e ".[evals]" transformers`): the extra provides the clients the eval suites use (for example `openai` and `tiktoken` for mmlu_pro), and `transformers` reads the model's tokenizer (`--tokenizer`). This builds and links the `gbench` CLI executable without installing development tools. The [installation guide](../../docs/installation.md) lists every optional install.

In [ ]:
import os, sys
from pathlib import Path

# Safe environment setup: Always normalize to top-level repository
if Path("/content").exists():
    %cd -q /content
    if not Path("/content/gbench").is_dir():
        !git clone https://github.com/google-gemma/gbench.git
    %cd -q /content/gbench
else:
    # Use the gbench checkout you are in (also when this notebook is opened from
    # examples/notebooks/), else clone one next to it.
    root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "pyproject.toml").is_file() and (p / "gbench" / "cli.py").is_file()), None)
    if root is None:
        if not Path("gbench/pyproject.toml").is_file():
            !git clone https://github.com/google-gemma/gbench.git
        root = Path("gbench").resolve()
    %cd -q {root}

%pip install -e ".[evals]" transformers -q
import gbench
print(f"gbench version {gbench.__version__} installed successfully.")

# Inspect available evaluation benchmark suites and pillars
!gbench --list evals
!gbench --list pillars

## Hugging Face authentication (required)

This notebook downloads the Gemma 4 GGUF (and its vision projector) plus tokenizers and eval datasets (e.g. GPQA) from the Hugging Face Hub with `huggingface_hub`. Some are **gated**, so an **`HF_TOKEN` is required**.

1. Create a **read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and accept the license on any gated model/dataset page you use (e.g. GPQA).
2. On **Colab**: click the **key icon (Secrets)** in the left sidebar, then **Add new secret**, name it `HF_TOKEN`, paste the token, and toggle **Notebook access** on.
3. **Elsewhere**: set it in your environment, e.g. `export HF_TOKEN=hf_...` (or `os.environ["HF_TOKEN"] = "hf_..."`).

The next cell loads the token and stops with instructions if it is missing.

In [ ]:
import os

# HF_TOKEN is REQUIRED: this notebook downloads the Gemma 4 GGUF (+ vision projector),
# tokenizers, and eval datasets (e.g. GPQA) from the Hugging Face Hub via
# huggingface_hub, which authenticates with it (resumable, higher rate limits, and
# access to gated repos).
try:
    from google.colab import userdata          # Colab: read from the Secrets vault
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass                                        # not on Colab, or the secret is unset
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError(
        "HF_TOKEN is not set. On Colab: click the key icon (Secrets) in the left "
        "sidebar, add a secret named HF_TOKEN, and turn on notebook access. "
        "Elsewhere: os.environ['HF_TOKEN'] = 'hf_...'. "
        "Create a read token at https://huggingface.co/settings/tokens."
    )
print("HF_TOKEN loaded.")

## 2. Installing Ollama locally

We check if the Ollama binary is present on the system. If it is not found, we install Ollama using its official Linux installation script (`curl -fsSL https://ollama.com/install.sh | sh`). Finally, we run `ollama --version` to verify that the installation succeeded and the CLI is available.

In [ ]:
import subprocess, os, shutil, glob

# (Re)install Ollama unless both the binary and its llama-server runner are present.
def _ollama_ready():
    if not shutil.which("ollama"):
        return False
    return any(glob.glob(p) for p in (
        "/usr/local/lib/ollama/llama-server",
        "/usr/local/lib/ollama/*/llama-server",
        "/usr/lib/ollama/llama-server",
    ))

if not _ollama_ready():
    print("Installing/repairing Ollama (binary + llama-server runner)...")
    # Ensure zstd is available (required by Ollama Linux tar.zst packages)
    subprocess.run("command -v zstd >/dev/null || (command -v apt-get >/dev/null && apt-get update -qq && apt-get install -y -qq zstd)", shell=True)
    # Run official Ollama installer
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
else:
    print("Ollama (with llama-server runner) already installed.")

# Ensure binary directory is present in PATH for subsequent cells
for p in ["/usr/local/bin", "/usr/bin", os.path.expanduser("~/.local/bin")]:
    if os.path.exists(os.path.join(p, "ollama")) and p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"

!ollama --version

## 3. Launching background Ollama server

We launch the `ollama serve` process in the background and send a health check request to `http://localhost:11434/` to verify that the HTTP API is alive ("Ollama is running").

In [ ]:
import subprocess, time, requests
try:
    resp = requests.get("http://localhost:11434/", timeout=2)
    print("Ollama server already active:", resp.text.strip())
except Exception:
    print("Starting background ollama serve...")
    OLLAMA_PROC = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL,
                                   stderr=subprocess.DEVNULL)   # stopped in the cleanup cell
    time.sleep(4)
    resp = requests.get("http://localhost:11434/")
    print("Server health check:", resp.text.strip())

## 4. Downloading the GGUF and writing the Modelfile

We download the quantized Gemma 4 GGUF **and its vision projector (`mmproj`)** from the Hugging Face Hub with `huggingface_hub` (authenticated via `HF_TOKEN`, so the download is resumable and not rate-limited), then write an Ollama `Modelfile.qat` that points `FROM` the **local** files:
* **`num_ctx 40960`**: context window of 40960 tokens. The window bounds prompt + reply and Ollama does not report it to gbench, so it must hold the largest request: `mmlu_pro` asks for up to 32768 reply tokens after a 5-shot prompt. The optional thinking run needs a larger window (65536 reply tokens); its cell shows how to raise it. If Ollama cannot allocate the window, use a machine with more memory rather than a smaller window: a smaller window cuts long replies.

The two `FROM` lines (main GGUF + `mmproj`) keep the model's **vision** capability. If `ollama create` in the next cell reports a missing file, re-run this cell. If you prefer vLLM over Ollama, skip this section and serve with `vllm serve google/gemma-4-26B-A4B-it --port 8000 --reasoning-parser gemma4` (26B-A4B in bf16 needs a large datacenter GPU; on a smaller GPU serve `google/gemma-4-E4B-it`), then pass `--remote-endpoint http://127.0.0.1:8000/v1 --tokenizer <the model id you served>` (e.g. `google/gemma-4-26B-A4B-it`) to `gbench`.

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download

HF_REPO = "unsloth/gemma-4-E4B-it-qat-GGUF"
HF_QUANT = "UD-Q4_K_XL"
token = os.environ["HF_TOKEN"]  # required; loaded in the Hugging Face auth cell above

# Download the model GGUF and its vision projector (mmproj) with your HF_TOKEN, then
# build the Modelfile from the local files (the second FROM line keeps vision).
files = HfApi().list_repo_files(HF_REPO, token=token)
main = [f for f in files if f.endswith(".gguf") and HF_QUANT in f]
proj = [f for f in files if f.endswith(".gguf") and "mmproj" in f.lower() and "-F16" in f]
if not main:
    raise RuntimeError(f"No {HF_QUANT} .gguf found in {HF_REPO}.")
GGUF_PATH = os.path.realpath(hf_hub_download(HF_REPO, main[0], token=token))
lines = [f"FROM {GGUF_PATH}"]
if proj:  # vision projector -> keeps multimodal capability
    lines.append(f"FROM {os.path.realpath(hf_hub_download(HF_REPO, proj[0], token=token))}")
lines += ['PARAMETER num_ctx 40960']
with open("Modelfile.qat", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print("Created Modelfile.qat from local GGUF" + (" + mmproj (vision)" if proj else ""))

## 5. Registering model and running generation smoke test

We register our custom model tag (`gemma4-qat:4b`) with `ollama create -f Modelfile.qat`. Because `Modelfile.qat` points `FROM` the local GGUF (and `mmproj`) downloaded in the previous cell, this reads from disk, with no network pull. We then run a quick generation test to verify the model loads into hardware memory and generates tokens correctly.

In [ ]:
import subprocess, requests

MODEL_TAG = "gemma4-qat:4b"
print(f"Registering model {MODEL_TAG} from the local GGUF (built in the previous cell)...")
subprocess.run(["ollama", "create", MODEL_TAG, "-f", "Modelfile.qat"], check=True)

print("Running quick generation smoke test via Ollama API (cold load into GPU VRAM)...")
resp = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL_TAG, "prompt": "Reply with the single word: READY.", "stream": False},
    timeout=300,
)
print("Smoke test response:", resp.json().get("response", "").strip())

## 6. Verifying OpenAI REST endpoint readiness

Before launching `gbench`, we query `http://localhost:11434/v1/models` to verify that Ollama is serving standard OpenAI `/v1` REST payloads and that our registered model is listed.

In [ ]:
import requests
resp = requests.get("http://localhost:11434/v1/models")
print("OpenAI /v1/models endpoint HTTP status:", resp.status_code)
models = [m["id"] for m in (resp.json().get("data") or [])]
print("Available REST models:", models)
if not models:
    print("No models registered yet - re-run the model registration cell above.")

## 7. Running academic benchmarks

The setup cell already listed every registered suite (`!gbench --list evals`); `!gbench --list pillars` lists their pillars.

> **Recommended: run evals with thinking (`--eval-thinking`).** That measures the model as it reasons. This notebook runs **no-think** only to keep the smoke test fast; the thinking run is the optional cell below.

Then we run `gbench --evals-only` on `gpqa_diamond` (graduate-level science) and `mmlu_pro` (multi-domain reasoning). To keep the notebook runnable on a laptop/Colab box, the run below is **no-think** and uses `--eval-limit 20` (20 questions **per suite**), a subset gbench flags as non-leaderboard-comparable. (`gpqa_diamond` follows simple-evals and asks each question 10 times with shuffled options, so 20 questions are 200 generations; its correct/total counts those copies.) `--max-output-tokens` is **required** for any eval run.

Ollama may not honour gbench's no-think switch (vLLM does), so a no-think run on Ollama can still contain reasoning: each sample in the result's `sample_traces` has a `reasoning` field, which should be empty (null) on a no-think run. For a guaranteed switch, use the vLLM path.

To measure the model's **reasoning** (Chain-of-Thought), the recommended setting for real eval runs, use the thinking cell below; it is commented out so the smoke test stays fast.

In [ ]:
# Academic evals, NO-THINK and limited to a small subset so the notebook runs on
# modest hardware. --eval-limit 20 runs 20 questions PER SUITE; --max-output-tokens
# is REQUIRED for any eval run (no-think gpqa_diamond and mmlu_pro send their own
# official caps instead; see max_output_tokens_note in each result). gbench flags a
# subset run as non-leaderboard-comparable.
# (To measure the model's reasoning instead, see the optional thinking cell below.)
# --models names the tag Ollama serves (MODEL_TAG, registered above).
!gbench --evals-only \
        --remote-endpoint http://localhost:11434/v1 \
        --models {MODEL_TAG} \
        --tokenizer google/gemma-4-E4B-it \
        --evals gpqa_diamond mmlu_pro \
        --max-output-tokens 65536 \
        --eval-limit 20 \
        --results-dir ./results_evals

# --- Alternative: serve with vLLM instead of Ollama (pass the model id you served as --tokenizer) ---
# In another shell:  vllm serve google/gemma-4-26B-A4B-it --port 8000 --reasoning-parser gemma4
#   !gbench --evals-only \
#           --remote-endpoint http://127.0.0.1:8000/v1 \
#           --tokenizer google/gemma-4-26B-A4B-it \
#           --evals gpqa_diamond mmlu_pro \
#           --max-output-tokens 65536 \
#           --eval-limit 20 \
#           --results-dir ./results_evals

### Recommended for real eval runs: reasoning (thinking) mode

The runs in this notebook are **no-think** on a small subset only so the smoke test is fast on modest hardware. For real eval numbers, run with thinking (`--eval-thinking`): it measures the model as it reasons (Chain-of-Thought), which takes longer. The next cell shows how; it is **commented out** so nothing heavy runs by accident. Uncomment it (and keep `--eval-limit` small, e.g. 5) to measure reasoning mode.

In [ ]:
# OPTIONAL: reasoning (thinking) mode - HEAVIER, so it is commented out and does NOT
# run by default. It measures the model's Chain-of-Thought (a different, slower thing
# than the no-think run above). Uncomment to run it, and keep --eval-limit small
# (e.g. 5) because reasoning chains are long on modest hardware.
# The 65536-token budget needs a larger Ollama window than the 40960 above: raise it first.
# !sed -i 's/^PARAMETER num_ctx .*/PARAMETER num_ctx 73728/' Modelfile.qat && ollama create {MODEL_TAG} -f Modelfile.qat
# !gbench --evals-only \
#         --remote-endpoint http://localhost:11434/v1 \
#         --models {MODEL_TAG} \
#         --tokenizer google/gemma-4-E4B-it \
#         --evals gpqa_diamond mmlu_pro \
#         --eval-thinking \
#         --max-output-tokens 65536 \
#         --eval-limit 5 \
#         --results-dir ./results_evals_cot

## 8. Loading and inspecting per-suite results

`gbench` writes one JSON file per suite into a timestamped subdirectory of `--results-dir`, named `eval_<suite>_<model>_<format>.json` (for this notebook, e.g. `eval_gpqa_diamond_gemma4-qat:4b_remote-endpoint.json`: the `--models` id, then the format). The cell below recursively globs those files from `./results_evals`, `./results_evals_cot` (the optional thinking run) and `./results_evals_mm` (section 9; run that section first to include it) and prints each suite's headline accuracy, so you can compare suites and thinking modes.

In [ ]:
import json
from pathlib import Path

roots = [Path("./results_evals"), Path("./results_evals_cot"), Path("./results_evals_mm")]
result_files = [f for root in roots for f in sorted(root.rglob("eval_*.json"))]
if not result_files:
    print("No eval_*.json files found under ./results_evals* - run section 7 first.")
for f in result_files:
    data = json.loads(f.read_text())
    acc = data.get("accuracy")
    # accuracy is already a percentage (0-100), so print it as-is.
    acc_str = f"{acc:.1f}%" if isinstance(acc, (int, float)) else "n/a"
    lines = [f.name,
             f"  status:          {data.get('status', 'ok')}",
             f"  thinking:        {data.get('thinking', False)}"]
    if data.get("correct_answers") is not None:     # accuracy-only suites (mmmu_pro) have no count
        reps = data.get("repeats")
        lines.append(f"  correct/total:   {data['correct_answers']}/{data.get('total_questions', '?')}"
                     + (f" ({reps['questions']} questions x {reps['n_repeats']})"
                        if isinstance(reps, dict) else ""))
    lines.append(f"  accuracy:        {acc_str}")
    print("\n".join(lines) + "\n")


## 9. Running a multimodal vision benchmark

Vision suites such as `mmmu_pro` (and `chartqa`) test image understanding. The run below evaluates `mmmu_pro` through the same endpoint, **no-think** with `--eval-limit 20` to stay light on modest hardware, like the other runs.

**Soft-token budgeting:** the number of vision soft tokens a chart or figure is encoded into matters for accuracy. When you drive a server with `--remote-endpoint`, that budget is fixed **server-side** (Ollama/vLLM), so gbench rejects `--eval-max-soft-tokens` in that mode. The `--eval-max-soft-tokens` flag applies only to vision suites (`mmmu_pro`, `screenspot`, `screenspot_pointing`, `textvqa`, `infographicvqa`, `bundled_detection`) when gbench launches its **own local vLLM** server (no `--remote-endpoint`).

As before, `--max-output-tokens` is required.

In [ ]:
# Multimodal vision eval through the remote endpoint (soft tokens fixed server-side,
# so --eval-max-soft-tokens is NOT passed here). No-think + a small subset
# (--eval-limit 20) so it runs on modest hardware. --max-output-tokens is required.
!gbench --evals-only \
        --remote-endpoint http://localhost:11434/v1 \
        --models {MODEL_TAG} \
        --tokenizer google/gemma-4-E4B-it \
        --evals mmmu_pro \
        --max-output-tokens 4096 \
        --eval-limit 20 \
        --results-dir ./results_evals_mm

# --- Local vLLM path (gbench launches its own server) where --eval-max-soft-tokens applies ---
# Omit --remote-endpoint so gbench starts vLLM itself (it needs --models and --num-gpus),
# then you may budget soft tokens. It needs vllm (and its torch/CUDA) installed yourself
# (pip install vllm; gbench does not install it), and 26B-A4B in bf16 needs a large GPU:
#   !gbench --evals-only \
#           --models google/gemma-4-26B-A4B-it \
#           --num-gpus 1 \
#           --evals mmmu_pro \
#           --eval-max-soft-tokens 1120 \
#           --max-output-tokens 4096 \
#           --eval-limit 20 \
#           --results-dir ./results_evals_mm

## 10. Session cleanup and server shutdown

We unload the model, stop the Ollama server if this notebook started it, and remove the temporary Modelfile.

In [ ]:
import subprocess, os

# Unload the model, then stop the Ollama server only if this notebook started it
# (a server you were already running is left running).
subprocess.run(["ollama", "stop", "gemma4-qat:4b"], check=False)
if "OLLAMA_PROC" in globals():
    OLLAMA_PROC.terminate()
    OLLAMA_PROC.wait(timeout=30)
    print("Stopped the Ollama server this notebook started.")
else:
    print("Ollama server left running (it was not started by this notebook).")
if os.path.exists("Modelfile.qat"):
    os.remove("Modelfile.qat")
print("Session cleanup complete.")